In [ ]:
import os
REPO_GIT = "68056069-M/bkk-condo-price-mlops.git"
GITHUB_TOKEN = os.getenv("GITHUB_TOKEN")
GITHUB_HTTPS = f"https://{GITHUB_TOKEN}@github.com/{REPO_GIT}"

# Clone git repository
!git clone $GITHUB_HTTPS

In [ ]:
# Changing directory to working directory
%cd bkk-condo-price-mlops

# Configure Git identity (session-only)
!git config --global user.name "Pemika"
!git config --global user.email "68056069@kmitl.ac.th"

In [ ]:
%pwd #PRINT WORKING DIRECTORY

In [ ]:
# Install and Initialize DVC # แบบเอา -q ออก

!pip install --quiet dvc[gdrive]
!dvc init -f

In [ ]:
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

# Create the DVC remote folder in Drive
remote_path = "/content/drive/MyDrive/condo_dvc"
os.makedirs(remote_path, exist_ok=True)
print(f"DVC remote folder exists at: {remote_path}")

# Register a new remote under DVC’s configuration ทำครั้งเดียว
!dvc remote add -d mydrive {remote_path}

In [ ]:
# Commit and push your DVC setup
!git add .dvc .gitignore
!git commit -m "chore: init DVC and configure GDrive remote"
!git push -u origin main

In [ ]:
import os
import pandas as pd
from sklearn import datasets

os.makedirs("data", exist_ok=True)


df=pd.read_csv('data/Bangkok Housing Condo Apartment Prices.csv')

print("Saved raw Iris data to data/Bangkok Housing Condo Apartment Prices.csv")

In [ ]:
df.head()

In [ ]:
!dvc add "data/Bangkok Housing Condo Apartment Prices.csv"

In [ ]:
# Stage the pointer file, .gitignore, and DVC config
!git add "data/Bangkok Housing Condo Apartment Prices.csv.dvc" data/.gitignore .dvc/config

# Commit with a clear message
!git commit -m "feat: add raw Iris dataset under DVC remote config"

# Push your commit up to GitHub
!git push

In [ ]:
# Upload the CSV blob into /content/drive/MyDrive/mlops-workshop
!dvc push

In [ ]:
# List item in data/ (on DVC)
!dvc list . data

In [ ]:
# Log Git commit
!git log --oneline

In [ ]:
# Install DVC with Google Drive support
!pip install --quiet dvc[gdrive]

# Check that your DVC remote is registered (should show 'mydrive')
!dvc remote list

# Load the dataset from remote DVC storage
!dvc pull        # Fetches Bangkok Housing Condo Apartment Prices.csv from Google Drive
!dvc checkout    # Restores Bangkok Housing Condo Apartment Prices.csv into data/ folder

In [ ]:
# List data files
!dvc list . data

In [ ]:
!pwd

In [ ]:
import pandas as pd

# Load the versioned Iris CSV
df = pd.read_csv("data/Bangkok Housing Condo Apartment Prices.csv")
print("Data shape:", df.shape)
print("Data head(5): ")
df.head()

In [ ]:
print("Data info:", df.info())

In [ ]:
# Install necessary libraries
!pip install --quiet mlflow seaborn

# Import for plotting
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
# Summary statistics
df.describe()

In [ ]:
import pandas as pd

# สมมติว่า df คือ DataFrame
# 1. ใช้ pd.get_dummies() เพื่อแปลงคอลัมน์ 'Property Type' เป็น One-Hot Encoding
df_encoded = pd.get_dummies(df, columns=['Property Type'], prefix='Type')

# 2. ตอนนี้คอลัมน์ 'Property Type' ถูกแทนที่ด้วยคอลัมน์ตัวเลขหลายคอลัมน์
#    เช่น 'Type_Condo', 'Type_Apartment'
#    คุณสามารถคำนวณสหสัมพันธ์ได้ตามปกติ โดยเลือกเฉพาะคอลัมน์ที่แปลงแล้วและคอลัมน์เป้าหมาย
#    **หมายเหตุ:** การคำนวณสหสัมพันธ์กับ One-Hot Encoding อาจให้ผลลัพธ์ที่ซับซ้อน ควรดูสหสัมพันธ์ระหว่าง Price กับแต่ละ Type แยกกัน



In [ ]:
df_encoded.head()

In [ ]:
df_encoded.columns

In [ ]:
df_encoded.shape

In [ ]:
sns.pairplot(df, hue="Location", corner=True)
plt.show()

In [ ]:
# Correlation heatmap
feat_corr = ['Price (THB)'] + [col for col in df_encoded.columns if col.startswith('Type_')]
feat_corr_encoded = df_encoded[feat_corr].corr()
plt.figure(figsize=(6,5))
sns.heatmap(feat_corr_encoded, annot=True, fmt=".2f", cmap="vlag")
plt.title("Feature Correlations (no target)")
plt.show()

# **Train**

In [ ]:
from sklearn.model_selection import train_test_split

# 1. กำหนด Features (X) และ Target (y)
X = df_encoded.drop(columns=["Price (THB)", "Location"])
y = df_encoded["Price (THB)"]

# 2. ทำ 80/20 split โดยไม่มี Stratification (ลบ stratify=y ออก)
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42 # กำหนด random_state เพื่อให้ผลลัพธ์ซ้ำได้
)

print(f"Train size: {X_train.shape}")
print(f"Test size: {X_test.shape}")

In [ ]:
import mlflow
import mlflow.sklearn
from mlflow.models.signature import infer_signature
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np
import pandas as pd # ต้องเพิ่ม import pandas เพื่อใช้ get_dummies

# ──────────────────────────────────────────────────────────────────────────────
# 0. Data Pre-processing (การแก้ไข Error: 'House' string to float)
# ──────────────────────────────────────────────────────────────────────────────

# สมมติว่า X_train, X_test, y_train, y_test ได้ถูกโหลดไว้แล้ว
# (คุณต้องแน่ใจว่า X_train และ X_test ได้รับการโหลดที่นี่)

# 1. ทำ One-Hot Encoding สำหรับคอลัมน์ที่เป็น object/string ใน X_train และ X_test
# โดยใช้เฉพาะคอลัมน์ที่มี dtype เป็น 'object' (มักเป็น string/ข้อความ)
X_train_encoded = pd.get_dummies(X_train, drop_first=True)
X_test_encoded = pd.get_dummies(X_test, drop_first=True)

# 2. ปรับคอลัมน์ของ X_test ให้ตรงกับ X_train
# นี่เป็นขั้นตอนสำคัญ เพื่อให้แน่ใจว่า X_train และ X_test มีฟีเจอร์ที่เหมือนกันเป๊ะ
train_cols = X_train_encoded.columns

# เพิ่มคอลัมน์ใน X_test ที่มีใน X_train แต่ไม่มีใน X_test (กำหนดค่าเป็น 0)
missing_cols = set(train_cols) - set(X_test_encoded.columns)
for c in missing_cols:
    X_test_encoded[c] = 0

# ลบคอลัมน์ใน X_test ที่ไม่มีใน X_train
extra_cols = set(X_test_encoded.columns) - set(train_cols)
X_test_encoded = X_test_encoded.drop(columns=list(extra_cols), errors='ignore')

# จัดเรียงคอลัมน์ของ X_test ให้เหมือนกับ X_train
X_test_encoded = X_test_encoded[train_cols]

# ──────────────────────────────────────────────────────────────────────────────
# 1. Experiment & Run Setup
# ──────────────────────────────────────────────────────────────────────────────
mlflow.set_experiment("bkk_condo_price_regression")

dataset_name = "bkk_condo_prices"
dataset_version = "v1"
run_name = f"{dataset_name}-rf-{dataset_version}-encoded" # เปลี่ยนชื่อ run เล็กน้อยเพื่อสะท้อนการแก้ไข

with mlflow.start_run(run_name=run_name) as run:
    # Log experiment-wide parameters
    n_estimators = 100
    max_depth = 10

    mlflow.log_param("model_type", "RandomForestRegressor")
    mlflow.log_param("n_estimators", n_estimators)
    mlflow.log_param("max_depth", max_depth)
    mlflow.log_param("dataset", dataset_name)
    mlflow.log_param("author", dataset_name)
    mlflow.log_param("data_encoding", "OneHotEncoding_Pandas_GetDummies") # เพิ่ม log param การแก้ไข

    # ──────────────────────────────────────────────────────────────────────────
    # 2. Train (ใช้ข้อมูลที่ถูกเข้ารหัสแล้ว)
    # ──────────────────────────────────────────────────────────────────────────
    model = RandomForestRegressor(
        n_estimators=n_estimators,
        max_depth=max_depth,
        random_state=42)

    # ใช้ข้อมูลที่ถูกเข้ารหัสแล้ว: X_train_encoded
    model.fit(X_train_encoded, y_train)

    # ──────────────────────────────────────────────────────────────────────────
    # 3. Predict & Evaluate (ใช้ข้อมูลที่ถูกเข้ารหัสแล้ว)
    # ──────────────────────────────────────────────────────────────────────────
    # ใช้ข้อมูลที่ถูกเข้ารหัสแล้ว: X_test_encoded
    preds = model.predict(X_test_encoded)

    # เปลี่ยนเป็น Regression Metrics
    mse = mean_squared_error(y_test, preds)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test, preds)
    r2 = r2_score(y_test, preds)

    print(f"Test R2 Score: {r2:.4f}")
    print(f"Test RMSE: {rmse:.2f} THB")
    print(f"Test MAE: {mae:.2f} THB")

    # Log evaluation metrics to MLflow
    mlflow.log_metrics({
        "R2_score": r2,
        "RMSE": rmse,
        "MAE": mae
    })

    # ──────────────────────────────────────────────────────────────────────────
    # 4. Log Model to MLflow
    # ──────────────────────────────────────────────────────────────────────────
    # infer_signature ใช้ X_train ที่ถูกเข้ารหัสแล้ว
    signature = infer_signature(X_train_encoded, model.predict(X_train_encoded))
    input_example = X_train_encoded.head(1)

    mlflow.sklearn.log_model(
        sk_model = model,
        artifact_path = "random_forest_model",
        registered_model_name = f"{dataset_name}_rf_model",
        signature = signature,
        input_example = input_example
    )

    print(f"Run logged under '{run.info.run_name}' (ID: {run.info.run_id})")

In [ ]:
import mlflow
from mlflow.tracking import MlflowClient

# เปลี่ยนชื่อ Experiment จาก "iris_classification" เป็นชื่อของโปรเจกต์คอนโด
new_experiment_name = "bkk_condo_price_regression"

# Get experiment ID by name
exp = mlflow.get_experiment_by_name(new_experiment_name)

# ตรวจสอบว่าพบ Experiment หรือไม่ก่อนดำเนินการต่อ
if exp is None:
    print(f"Error: Experiment named {new_experiment_name!r} not found.")
else:
    print(f"Experiment name: {exp.name!r} → ID: {exp.experiment_id}")

    # Create client and list runs
    client = MlflowClient()

    # ดึงข้อมูล Runs ทั้งหมดใน Experiment นี้
    runs = client.search_runs(experiment_ids=[exp.experiment_id])

    # แสดงข้อมูลของแต่ละ Run
    for run in runs:
        print(f"Run ID: {run.info.run_id}")
        print(f"\tRun Name: {run.info.run_name}")
        print(f"\tParams: {run.data.params}")

        # สำหรับงาน Regression, Metrics ควรเป็น RMSE, MAE, R2_score เป็นต้น
        print(f"\tMetrics: {run.data.metrics}")

In [ ]:
import os
# Install MLflow and ngrok
!pip install mlflow pyngrok --quiet

# Import and set your ngrok authtoken
from pyngrok import ngrok

NGROK_AUTH_TOKEN = os.getenv("NGROK_AUTH_TOKEN")  # paste your ngrok token
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

# Spin up the MLflow UI on port 5000
import subprocess, time
subprocess.Popen(
    ["mlflow", "ui", "--host", "0.0.0.0", "--port", "5000"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.STDOUT
)
time.sleep(5)

# Open the public ngrok tunnel
public_url = ngrok.connect(5000)
print("MLflow UI is live at:", public_url)

In [ ]:
# Define source and target paths
import os, shutil

src = '/content/bkk-condo-price-mlops' # adjust this path to clone project on you colab env session
dst = f'/content/drive/MyDrive/mlops-workshop/mlruns-{dataset_version}'  # adjust this path to wherever you like in your Drive

# Create the target directory if it doesn't already exist
if not os.path.exists(dst):
    os.makedirs(dst, exist_ok=True)
    print(f"Created directory: {dst}")
else:
    print(f"Directory already exists: {dst}")

# Copy the entire mlruns folder into Drive
shutil.copytree(src, dst, dirs_exist_ok=True)
print(f"Copied `{src}` → `{dst}`")

In [ ]:
# Create a models/ folder
import os
import joblib

os.makedirs("models", exist_ok=True)

# Save as .pkl (single file)
model_name = f"{dataset_name}-{dataset_version}.pkl"
model_path = f"models/{model_name}"

joblib.dump(model, model_path)
print(f"Model saved to {model_path}")

In [ ]:
# Track the model with DVC
!dvc add models/{dataset_name}-{dataset_version}.pkl

# Verify DVC tracked it
!dvc list . models

In [ ]:
# Git add the .dvc pointer and updated .gitignore
!git add models/{dataset_name}-{dataset_version}.pkl.dvc models/.gitignore

# Commit and push the pointer to GitHub
!git commit -m "feat: save baseline KNN model with joblib and DVC"
!git push

In [ ]:
# View the commit log
!git log --oneline

# Upload the actual .pkl file to the DVC remote
!dvc push

In [ ]:
# Install required packages
!pip install mlflow pyngrok --quiet

# Mount your Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Configure MLflow to use your Drive-backed mlruns folder
import mlflow
DRIVE_MLRUNS = '/content/drive/MyDrive/mlops-workshop/mlruns-baseline'  # adjust as needed
mlflow.set_tracking_uri(f'file://{DRIVE_MLRUNS}')

In [ ]:
import os
# Configure ngrok
from pyngrok import ngrok
NGROK_AUTH_TOKEN = os.getenv("NGROK_AUTH_TOKEN")  # ← replace with your real token
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

# Launch the MLflow UI on port 5000 in the background
import subprocess, time
subprocess.Popen(
    ["mlflow", "ui",
     "--backend-store-uri", f"file://{DRIVE_MLRUNS}",
     "--host", "0.0.0.0",
     "--port", "5000"
    ],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.STDOUT
)
time.sleep(5)  # give it a moment to start

# Open an ngrok tunnel to expose port 5000
public_url = ngrok.connect(5000, "http")
print("MLflow UI is live at:", public_url)

In [ ]:
!ls -l models/

In [ ]:
# Load model using joblib (สมมติว่าคุณได้บันทึกโมเดล Random Forest Regression ที่ดีที่สุดไว้)
import joblib
try:
    model = joblib.load("models/bkk_condo_prices-v1.pkl")
    print("Model loaded successfully.")
except FileNotFoundError:
    print("Error: Model file 'models/condo-regression.pkl' not found.")
    print("Please ensure your best model is saved with the correct path.")
    exit()

import pandas as pd
import numpy as np

# ──────────────────────────────────────────────────────────────────────────
# 1. กำหนด Feature Names
# ──────────────────────────────────────────────────────────────────────────
# !!! สำคัญ: โปรดแทนที่รายการนี้ด้วยชื่อคอลัมน์ (Features) ที่ผ่านการทำความสะอาด/แปลงข้อมูล
# เช่น One-Hot Encoding หรือ Label Encoding แล้ว ที่โมเดลของคุณใช้ในการฝึก
# ตัวอย่างเช่น หาก Features คือ พื้นที่ (sqm) และ ประเภทห้อง (Condo=1, Apt=0)
cols = [
    "Area (sq. ft.)",
    "Bedrooms",
    "Bathrooms",
    "Type_Apartment",
    "Type_Condo",
    "Type_House"
]

# ──────────────────────────────────────────────────────────────────────────
# 2. เตรียมตัวอย่างข้อมูลนำเข้า (Sample Input)
# ──────────────────────────────────────────────────────────────────────────
# ตัวอย่าง Input: (ค่าตัวเลขต้องสอดคล้องกับ Features ใน cols)
# Example: 80 sqm, floor 15, 2 bedrooms, Is_Condo=1, Is_Apartment=0
sample_data = [
    [800.0, 3, 2, 0, 1, 0]
]

# Create one-row DataFrames with proper column names
sample_df = pd.DataFrame(sample_data, columns=cols)

# View sample structure
print("──────────────────────────────────────────────────────────────────────────")
print("Sample Input for Prediction:")
print(sample_df)
print("──────────────────────────────────────────────────────────────────────────")

# ──────────────────────────────────────────────────────────────────────────
# 3. Predict using the loaded model
# ──────────────────────────────────────────────────────────────────────────
predicted_price = model.predict(sample_df)[0]

# แสดงผลลัพธ์การทำนาย
print("Predicted Price (THB):")
print(f"฿ {predicted_price:,.2f}")

In [ ]:
TARGET = "Price (THB)"

# FEATURES: รายการคอลัมน์ Features ที่ทำความสะอาดและแปลงแล้ว
# *** แทนที่ด้วยรายการ Features ที่สมบูรณ์และถูกต้องตามที่คุณใช้ในการฝึกโมเดล ***
# (รวมถึง Features ที่ผ่าน One-Hot Encoding เช่น Type_Condo, Type_Apartment)
FEATURES = [
    "Area (sq. ft.)",
    "Bedrooms",
    "Bathrooms",
    "Type_Apartment",
    "Type_Condo",
    "Type_House"]

# เตรียมข้อมูล X และ y
df_processed = df_encoded.copy()
# *** ข้อควรระวัง: ต้องแน่ใจว่า df_processed มีการทำ Preprocessing (Cleaning, Encoding) แล้ว
# ถ้ายังไม่มีโค้ด Preprocessing อยู่ที่นี่ คุณต้องเพิ่มมันก่อนขั้นตอนถัดไป
# เช่น df_processed = pd.get_dummies(df_processed, columns=['Property Type'], prefix='Type') ***

X = df_processed[FEATURES]
y = df_processed[TARGET]

# ──────────────────────────────────────────────────────────────────────────
# 3. Simulate Two Time Periods (Reference vs Current)
# ──────────────────────────────────────────────────────────────────────────
# ใช้ train_test_split เพื่อแบ่งข้อมูล
# 1. ลบ stratify=y ออก เนื่องจากนี่คืองาน Regression (ค่าต่อเนื่อง)
# 2. ตั้งชื่อใหม่เพื่อให้สอดคล้องกับการจำลอง Reference (ข้อมูลเก่า) vs Current (ข้อมูลใหม่)
# Reference (70%) ใช้สำหรับสร้าง Base Model หรือเป็นช่วงอ้างอิง
# Current (30%) ใช้สำหรับจำลองข้อมูลที่เข้ามาใหม่เพื่อทำ Model Monitoring
ref_df, curr_df = train_test_split(
    df_processed, # แบ่ง DataFrame ที่มีทั้ง Features และ Target
    test_size=0.3,
    random_state=42,
    # ลบ stratify ออกสำหรับงาน Regression
)

print(f"\nReference Dataset Size (Data for initial training): {len(ref_df)} rows")
print(f"Current Dataset Size (Data for monitoring): {len(curr_df)} rows")

In [ ]:
import joblib
import pandas as pd
import numpy as np
import os

# ──────────────────────────────────────────────────────────────────────────
# 1. โหลดโมเดล Regression (ที่ฝึกด้วยข้อมูลคอนโด)
# ──────────────────────────────────────────────────────────────────────────
# *** ตรวจสอบและแก้ไขพาธนี้ให้ตรงกับพาธที่คุณบันทึกโมเดล Regression ที่ดีที่สุดของคุณ ***
MODEL_PATH = "models/bkk_condo_prices-v1.pkl"
try:
    model = joblib.load(MODEL_PATH)
    print(f"Model loaded successfully from: {MODEL_PATH}")
except FileNotFoundError:
    print(f"Error: Model file '{MODEL_PATH}' not found. Please check your path.")
    exit()

# ──────────────────────────────────────────────────────────────────────────
# 2. ทำนายและเพิ่มคอลัมน์ 'prediction'
# ──────────────────────────────────────────────────────────────────────────

# FEATURES ต้องเป็นรายการคอลัมน์ที่ถูกต้องและเรียงลำดับตามที่โมเดลเคยฝึก
FEATURES = [
    "Area (sq. ft.)",
    "Bedrooms",
    "Bathrooms",
    "Type_Apartment",
    "Type_Condo",
    "Type_House"]

# ทำนายผลลัพธ์บน Reference Dataset และเพิ่มคอลัมน์ "prediction"
ref_df["prediction"] = model.predict(ref_df[FEATURES])

# ทำนายผลลัพธ์บน Current Dataset และเพิ่มคอลัมน์ "prediction"
curr_df["prediction"] = model.predict(curr_df[FEATURES])

print("\n'prediction' column successfully added to ref_df and curr_df.")
print(f"Reference Head:\n{ref_df[['Price (THB)', 'prediction']].head(3)}")
print(f"Current Head:\n{curr_df[['Price (THB)', 'prediction']].head(3)}")

In [ ]:
# 1.1 ฝึกโมเดล (ใช้ X_train, y_train จาก ref_df)
from sklearn.ensemble import RandomForestRegressor
# ... (สมมติว่าคุณมีโค้ด Preprocessing ที่สร้าง FEATURES ที่ถูกต้องแล้ว)

TARGET = "Price (THB)"
FEATURES = ["Area (sq. ft.)", "Bathrooms", "Bedrooms", "Type_Apartment", "Type_Condo", "Type_House"]

# แยก Features และ Target
X_ref = ref_df[FEATURES]
y_ref = ref_df[TARGET]

model_ref = RandomForestRegressor(random_state=42)
model_ref.fit(X_ref, y_ref)

# 1.2 ทำนายผลลัพธ์บนชุดข้อมูล Reference และ Current
ref_df['prediction'] = model_ref.predict(X_ref)

# จำลองการทำนายบนชุดข้อมูลใหม่ (Current Data)
X_curr = curr_df[FEATURES]
curr_df['prediction'] = model_ref.predict(X_curr)

# Note: เนื่องจากชุดข้อมูล Current เป็นการจำลองข้อมูลใหม่
#       เราควรมีคอลัมน์ Price (THB) ของข้อมูลจริงใน curr_df ด้วย เพื่อประเมินความแม่นยำ
#       แต่สำหรับการทำ Data Drift Detection, การมีแค่ FEATURES และ PREDICTION ก็เพียงพอ

In [ ]:
#ไม่ต้องรัน uninstall evidently
#!pip uninstall evidently -y

In [ ]:
#ไม่ต้องรันใช้ clear evidently
!rm -rf /usr/local/lib/python3.12/dist-packages/evidently*

In [ ]:
!pip install --upgrade evidently==0.6.7

In [ ]:
import evidently as ev
from evidently.report import Report
from evidently.metric_preset import (
  DataDriftPreset,
  DataQualityPreset,
  TargetDriftPreset,
  RegressionPreset
)
from evidently import ColumnMapping

In [ ]:
FEATURES   = ['Area (sq. ft.)', 'Bathrooms', 'Bedrooms']
TARGET_COL = "Price (THB)"
PRED_COL   = "prediction"
CATE_COL = ['Type_Condo', 'Type_Apartment', 'Type_House']

mapping = ColumnMapping()
mapping.target = TARGET_COL
mapping.prediction = PRED_COL
mapping.numerical_features = FEATURES
mapping.categorical_features = CATE_COL



In [ ]:

# ──────────────────────────────────────────────────────────────────────────
# 2.2 สร้างรายงาน Data Drift Report
# ──────────────────────────────────────────────────────────────────────────
data_drift_report = ev.report.Report(metrics=[
    DataDriftPreset(),
])


# ──────────────────────────────────────────────────────────────────────────
# 2.3 สร้างรายงาน Regression Performance Report
# ──────────────────────────────────────────────────────────────────────────
regression_performance_report = ev.report.Report(metrics=[RegressionPreset(),])


print("Evidently Report objects created successfully without TypeError.")

In [ ]:
# Save the report to HTML
out = "predit_model_condo_report.html"
regression_performance_report.save_html(out)
print(f"Saved report → {out}")

# **FastAPI Endpoint Development**

In [ ]:
!pip install --quiet dvc[gdrive]
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# คำสั่งนี้จะดึงไฟล์ข้อมูลและไฟล์โมเดลทั้งหมดที่ถูกควบคุมด้วย DVC มา
!dvc pull

In [ ]:
# Install DVC
!pip install --quiet dvc[gdrive]

# Install ngrok, fastapi and uvicorn
!pip install --quiet pyngrok
!pip install fastapi uvicorn

In [ ]:
%cd /content/bkk-condo-price-mlops

In [ ]:
# FastAPI

from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from fastapi.staticfiles import StaticFiles
from pydantic import BaseModel, Field
from typing import Literal
import joblib
import pandas as pd
import os

# Load model path from environment variable or default
MODEL_PATH = os.getenv("MODEL_PATH", "models/bkk_condo_prices-v1.pkl")

app = FastAPI(
    title="Bangkok Property Price Prediction API",
    version="1.0.0",
    description="Predicting property rental prices in Bangkok"
)

# Enable CORS for frontend or external access
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

# Serve static HTML frontend
# app.mount("/", StaticFiles(directory="frontend", html=True), name="static")

# Load model at startup
try:
    model = joblib.load(MODEL_PATH)
    print(f"✅ Model loaded from {MODEL_PATH}")
except Exception as e:
    raise RuntimeError(f" Failed to load the prediction model from {MODEL_PATH}: {e}")

# Input schema
class CondoInputData(BaseModel):
    area_sqft: float = Field(..., gt=0)
    no_bedroom: int = Field(..., ge=1)
    no_bathroom: int = Field(..., ge=1)
    property_type: Literal['Apartment', 'Condo', 'House']

# Root endpoint (optional, since frontend is served here)
@app.get("/api", summary="Root API endpoint")
def root():
    return {"message": "Welcome to the Bangkok Property Price Prediction API"}

# Health check
@app.get("/health", summary="API health check")
def health_check():
    return {
        "status": "ok",
        "model_loaded": True,
        "model_path": MODEL_PATH
    }

# Prediction endpoint
@app.post("/predict", summary="Predict condo rental price in THB")
def predict_price(data: CondoInputData):
    expected_columns = [
        "Area (sq. ft.)", "Bedrooms", "Bathrooms",
        "Type_Apartment", "Type_Condo", "Type_House"
    ]
    input_data = {
        "Area (sq. ft.)": data.area_sqft,
        "Bedrooms": data.no_bedroom,
        "Bathrooms": data.no_bathroom,
        "Type_Apartment": 0,
        "Type_Condo": 0,
        "Type_House": 0,
    }
    ohe_col = f"Type_{data.property_type}"
    if ohe_col in input_data:
        input_data[ohe_col] = 1

    input_df = pd.DataFrame([input_data], columns=expected_columns)

    try:
        prediction = model.predict(input_df)[0]
        return {
            "predicted_price_thb": max(0, round(prediction)),
            "message": f"Predicted price for a {data.property_type} of {data.area_sqft} sq.ft."
        }
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"Prediction failed: {str(e)}")

In [ ]:

%%writefile app.py

from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel, Field
from typing import Literal # Used for property_type validation
import joblib
import numpy as np
import pandas as pd
import os

# --- Configuration ---
# File path to the trained model artifact
MODEL_PATH = "models/bkk_condo_prices-v1.pkl"

app = FastAPI(
    title="Bangkok Property Price Prediction API",
    version="1.0.0",
    description="Predicting property rental prices in Bangkok"
)

# Enable CORS for all origins (needed for the Web Demo HTML file)
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

# ──────────────────────────────────────────────────────────────────────────────
# 1. Load the Model at Startup
# ──────────────────────────────────────────────────────────────────────────────
try:
    model = joblib.load(MODEL_PATH)
    print(f"Model loaded successfully from {MODEL_PATH}")
except Exception as e:
    raise RuntimeError(f"Failed to load the prediction model from {MODEL_PATH}: {e}")

# ──────────────────────────────────────────────────────────────────────────────
# 2. Request Schema (Input Features from User)
# ──────────────────────────────────────────────────────────────────────────────
class CondoInputData(BaseModel):
    """Defines the structure and validation for incoming condo features."""
    area_sqft: float = Field(
        ..., gt=0, description="Area in square feet (sq.ft.)"
    )
    no_bedroom: int = Field(
        ..., ge=1, description="Number of bedrooms (must be >= 1)"
    )
    no_bathroom: int = Field(
        ..., ge=1, description="Number of bathrooms (must be >= 1)"
    )
    property_type: Literal['Apartment', 'Condo', 'House'] = Field(
        ..., description="Type of property (Apartment, Condo, or House)"
    )

# ──────────────────────────────────────────────────────────────────────────────
# 3. Health‐check endpoint
# ──────────────────────────────────────────────────────────────────────────────
@app.get("/health", summary="API health check")
def health_check():
    """Checks if the API is running and the model is loaded."""
    return {
        "status": "ok",
        "model_loaded": True,
        "model_path": MODEL_PATH,
    }

# ──────────────────────────────────────────────────────────────────────────────
# 4. Prediction Endpoint (Main Logic)
# ──────────────────────────────────────────────────────────────────────────────
@app.post("/predict", summary="Predict condo rental price in THB")
def predict_price(data: CondoInputData):
    """
    Receives condo features and returns the predicted rental price.
    Performs One-Hot Encoding internally for property_type.
    """

    # 4.1 Define the 6 expected feature names in the EXACT order the model was trained with
    expected_columns = [
        "Area (sq. ft.)",
        "Bedrooms",
        "Bathrooms",
        "Type_Apartment",
        "Type_Condo",
        "Type_House"
    ]

    # 4.2 Initialize a DataFrame row with user's numerical inputs
    input_data = {
        "Area (sq. ft.)": data.area_sqft,
        "Bedrooms": data.no_bedroom,
        "Bathrooms": data.no_bathroom,
        # Initialize OHE columns to 0
        "Type_Apartment": 0,
        "Type_Condo": 0,
        "Type_House": 0,
    }

    # 4.3 Perform One-Hot Encoding (OHE) based on user input
    # Set the corresponding 'Type_' column to 1
    ohe_col_name = f"Type_{data.property_type}"
    if ohe_col_name in input_data:
        input_data[ohe_col_name] = 1

    # Create DataFrame, ensuring the columns are in the EXACT expected order
    input_df = pd.DataFrame([input_data], columns=expected_columns)

    try:
        # 4.4 Get prediction (Regression output is a single float)
        prediction = model.predict(input_df)[0]

        # Ensure prediction is positive and round to nearest whole number
        predicted_price = max(0, round(prediction))

        return {
            "predicted_price_thb": predicted_price,
            "message": f"Predicted price for a {data.property_type} of {data.area_sqft} sq.ft."
        }

    except Exception as e:
        raise HTTPException(
            status_code=500,
            detail=f"Prediction failed due to internal model error: {str(e)}"
)


#** Run FastAPI with Uvicorn and Expose via Ngrok **

In [ ]:
run_port = 8000
!nohup uvicorn app:app --host 0.0.0.0 --port {run_port} > uvicorn.log 2>&1 &

In [ ]:
# Check API is working
!curl -m 5 http://127.0.0.1:{run_port}/health

In [ ]:
import os
from pyngrok import ngrok
NGROK_AUTH_TOKEN = os.getenv("NGROK_AUTH_TOKEN")  # ← replace with your real token
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

public_url = ngrok.connect(run_port, "http")
print("FastAPI is live at:", public_url)

In [ ]:
!curl -v -X POST http://127.0.0.1:8000/predict \
     -H "Content-Type: application/json" \
     -d '{"area_sqft": 600.0, "Bedrooms": 2, "Bathrooms": 2, "property_type": "Condo"}'

In [ ]:
!tail -n 50 uvicorn.log

In [ ]:
# Check version of each package (prepare for docker)
!pip show fastapi
!pip show uvicorn
!pip show joblib
!pip show pandas
!pip show numpy
!pip show scikit-learn

In [ ]:
%%writefile Dockerfile
# Use an official Python runtime as a parent image
FROM python:3.12-slim

# Set the working directory in the container
WORKDIR /app

# Copy the current directory contents into the container at /app
COPY app.py .
COPY models /app/models

# Install any needed packages specified in requirements.txt
# Make sure to create a requirements.txt file in your project
# RUN pip install --no-cache-dir -r requirements.txt

# If you don't have a requirements.txt, you can install the required packages directly:
RUN pip install --no-cache-dir fastapi uvicorn joblib pandas numpy scikit-learn

# Expose the port the app runs on
EXPOSE 8000

# Run the uvicorn server when the container launches
CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"]

In [ ]:
!pip freeze > requirements.txt